# NB3 — ERA5 + NASA Alignment & Preprocessing (Single Year Template)
#
**Pipeline xử lý theo từng năm đơn lẻ nhằm tối ưu bộ nhớ Kaggle (Giới hạn 19.5GB):**
1. Load & merge các tập dữ liệu ERA5 (pressure, accum, instant) của duy nhất năm `TARGET_YEAR`.
2. Tái cấu trúc và ánh xạ bộ Normalization Stats toàn cục (đã được tính trước từ tập Train 2020-2024 để đảm bảo tính nhất quán của mô hình).
3. Chuẩn hóa (Normalize) dữ liệu của năm mục tiêu.
4. Tạo nhãn tương lai Y = dịch chuyển -1 timestep (Dự báo trước +6h).
5. Cắt lưới và nội suy không gian/thời gian cho dữ liệu NASA Tb khớp với ERA5.
6. Lưu cấu trúc Zarr cục bộ theo định dạng: `brain_{year}_x.zarr`, `brain_{year}_label.zarr` và `eye_{year}_norm.zarr`.
7. Tiến hành Sanity Check tính toàn vẹn dữ liệu.

In [1]:
!pip install zarr xarray dask netCDF4 h5netcdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.0 MB/s eta 0:00:00


In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import os, shutil, gc, json, glob

# ── CẤU HÌNH NĂM MỤC TIÊU (Thay đổi giá trị này cho từng phiên bản sao chép) ──
TARGET_YEAR = 2025  # Lần lượt đổi thành 2021, 2022, 2023, 2024, 2025, 2026

# ── ĐƯỜNG DẪN INPUT DATASET ──────────────────────────────────────────────────
BASE_ERA5 = "/kaggle/input/notebooks/b22dccn320trnminhhiu/fork-of-nb1-era5-2020-2026"
BASE_NASA = "/kaggle/input/notebooks/b22dccn320trnminhhiu/fork-of-nb2-nasa"

# Đường dẫn trỏ tới thư mục chứa file stats chung (Lưu ý: Bạn cần chạy tính toán trước 
# tập Train 2020-2024 một lần duy nhất, lưu lại file .json và .nc rồi upload thành Dataset riêng)
STATS_DIR = "/kaggle/input/notebooks/phongngtun/stat-compute" 

# ── ĐƯỜNG DẪN OUTPUT ─────────────────────────────────────────────────────────
OUT = "/kaggle/working"

# Cấu hình Chunk tối ưu hóa tính toán Lazy và dung lượng RAM trên Kaggle
ERA5_LOAD_CHUNKS = {'time': 100}
NASA_LOAD_CHUNKS = {'time': 100}
SAVE_CHUNKS = {'time': 100, 'lat': 50, 'lon': 50}

print(f"✓ Cấu hình hoàn tất cho NĂM MỤC TIÊU: {TARGET_YEAR}")

✓ Cấu hình hoàn tất cho NĂM MỤC TIÊU: 2025


In [3]:
print(f"--- Load ERA5 (lazy) từ thư mục của năm {TARGET_YEAR} ---")

PRESSURE_FILES = f"{BASE_ERA5}/era5_data/year_{TARGET_YEAR}/era5_pressure_*.nc"
ACCUM_FILES = f"{BASE_ERA5}/era5_data/year_{TARGET_YEAR}/*accum.nc"
INSTANT_FILES = f"{BASE_ERA5}/era5_data/year_{TARGET_YEAR}/*instant.nc"

# 1. Đọc dữ liệu các tầng áp suất khí quyển (Pressure levels)
pressure_paths = sorted(glob.glob(PRESSURE_FILES))
if not pressure_paths:
    raise FileNotFoundError(f"Không tìm thấy file pressure tương ứng với năm {TARGET_YEAR}")

ds_pressure = xr.open_mfdataset(pressure_paths, combine='by_coords',
                                 chunks=ERA5_LOAD_CHUNKS, engine='h5netcdf')
print(f"  Pressure level vars : {list(ds_pressure.data_vars)}")

# 2. Đọc dữ liệu lượng mưa tích tụ bề mặt (Surface Accum)
accum_paths = sorted(glob.glob(ACCUM_FILES))
ds_accum = xr.open_mfdataset(accum_paths, combine='by_coords',
                              chunks=ERA5_LOAD_CHUNKS, engine='h5netcdf')
print(f"  Accum vars          : {list(ds_accum.data_vars)}")

# 3. Đọc dữ liệu khí tượng tức thời bề mặt (Surface Instant)
instant_paths = sorted(glob.glob(INSTANT_FILES))
ds_instant = xr.open_mfdataset(instant_paths, combine='by_coords',
                                chunks=ERA5_LOAD_CHUNKS, engine='h5netcdf')
print(f"  Instant vars        : {list(ds_instant.data_vars)}")

--- Load ERA5 (lazy) từ thư mục của năm 2025 ---
  Pressure level vars : ['z', 'q', 'u', 'v']
  Accum vars          : ['tp']
  Instant vars        : ['u10', 'v10', 'd2m', 't2m', 'msl', 'z', 'lsm']


In [4]:
print(f"--- Merge các thành phần dữ liệu ERA5 năm {TARGET_YEAR} ---")

# Đồng bộ hóa tên biến để tránh xung đột địa thế 'z'
if 'z' in ds_instant.data_vars:
    ds_instant = ds_instant.rename({'z': 'geopotential_surf'})
    print("  Renamed instant z → geopotential_surf")
if 'z' in ds_accum.data_vars:
    ds_accum = ds_accum.rename({'z': 'geopotential_surf_acc'})
    print("  Renamed accum z → geopotential_surf_acc")

# Loại bỏ các tọa độ nhiễu hệ thống gây lỗi kết hợp
JUNK_COORDS = ['expver', 'step', 'number']
def drop_junk(ds):
    drop = [v for v in JUNK_COORDS if v in ds.coords or v in ds.data_vars]
    return ds.drop_vars(drop, errors='ignore')

ds_accum = drop_junk(ds_accum)
ds_instant = drop_junk(ds_instant)
ds_pressure = drop_junk(ds_pressure)

# Tiến hành hợp nhất dữ liệu trên trục thời gian giao thoa (join='inner')
ds_brain = xr.merge([ds_pressure, ds_accum, ds_instant], join='inner')

# Chuẩn hóa hệ thống nhãn tên tọa độ chiều không gian và thời gian
rename_map = {}
if 'valid_time' in ds_brain.coords: rename_map['valid_time'] = 'time'
if 'latitude'   in ds_brain.coords: rename_map['latitude']   = 'lat'
if 'longitude'  in ds_brain.coords: rename_map['longitude']  = 'lon'
if rename_map:
    ds_brain = ds_brain.rename(rename_map)
    print(f"  Renamed coords: {rename_map}")

print(f"\nThông tin bộ dữ liệu ds_brain năm {TARGET_YEAR}:")
print(f"  Các biến đặc trưng : {list(ds_brain.data_vars)}")
print(f"  Tổng số Timesteps  : {len(ds_brain.time)}")
print("✓ Hợp nhất dữ liệu hoàn tất.")

--- Merge các thành phần dữ liệu ERA5 năm 2025 ---
  Renamed instant z → geopotential_surf
  Renamed coords: {'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'}

Thông tin bộ dữ liệu ds_brain năm 2025:
  Các biến đặc trưng : ['z', 'q', 'u', 'v', 'tp', 'u10', 'v10', 'd2m', 't2m', 'msl', 'geopotential_surf', 'lsm']
  Tổng số Timesteps  : 1460
✓ Hợp nhất dữ liệu hoàn tất.


In [5]:
print("--- Nạp cấu trúc Normalization Stats toàn cục từ tập huấn luyện gốc ---")

with open(f'{STATS_DIR}/era5_normalization_stats.json', 'r') as f:
    stats_dict = json.load(f)

mean_vars = {}
std_vars = {}

# Phục hồi cấu trúc Dataset từ file JSON tĩnh để broadcast tính toán tối ưu
for var in ds_brain.data_vars:
    if 'pressure_level' not in ds_brain[var].dims:
        # Xử lý các biến 2D đặc trưng bề mặt
        mean_vars[var] = xr.DataArray(stats_dict[var]['mean'])
        std_vars[var] = xr.DataArray(stats_dict[var]['std'])
    else:
        # Xử lý các biến 3D phân tầng khí áp khí quyển
        levs = ds_brain.pressure_level.values
        m_vals = [stats_dict[f"{var}_{int(lev)}hpa"]['mean'] for lev in levs]
        s_vals = [stats_dict[f"{var}_{int(lev)}hpa"]['std'] for lev in levs]
        
        mean_vars[var] = xr.DataArray(m_vals, coords=[('pressure_level', levs)])
        std_vars[var]  = xr.DataArray(s_vals, coords=[('pressure_level', levs)])

era5_mean = xr.Dataset(mean_vars)
era5_std = xr.Dataset(std_vars)

print("✓ Trích xuất và xây dựng lại thành công bộ era5_mean và era5_std toàn cục.")

--- Nạp cấu trúc Normalization Stats toàn cục từ tập huấn luyện gốc ---
✓ Trích xuất và xây dựng lại thành công bộ era5_mean và era5_std toàn cục.


In [6]:
print(f"--- Thực hiện Normalize & Thiết lập Label Y cho năm {TARGET_YEAR} ---")

def normalize(ds, mean, std):
    """Phép tính Z-score chuẩn hóa lười biếng (lazy calculation)"""
    return (ds - mean) / std

# Thực hiện chuẩn hóa phi tập trung trên năm mục tiêu dựa theo stats huấn luyện chung
ds_norm = normalize(ds_brain, era5_mean, era5_std)

# Thiết lập chuỗi thời gian nhãn Y dịch bước thời gian t + 1 (Dự báo tương lai +6h)
ds_label = ds_norm.shift(time=-1).dropna(dim='time', how='all')

# Thu hẹp chiều thời gian tập X để đảm bảo tính đồng bộ tương quan thời gian
ds_x = ds_norm.sel(time=ds_label.time)

print(f"  Số mẫu dữ liệu X: {len(ds_x.time)} | Số mẫu dữ liệu Nhãn Y: {len(ds_label.time)}")
assert (ds_x.time.values == ds_label.time.values).all(), "LỖI HỆ THỐNG: Sai lệch đồng bộ trục thời gian giữa X và Y!"
print("  ✓ Đồng bộ liên kết ma trận thời gian X/Y hợp chuẩn.")

--- Thực hiện Normalize & Thiết lập Label Y cho năm 2025 ---
  Số mẫu dữ liệu X: 1459 | Số mẫu dữ liệu Nhãn Y: 1459
  ✓ Đồng bộ liên kết ma trận thời gian X/Y hợp chuẩn.


In [7]:
print(f"--- Tiến hành kết xuất luồng dữ liệu Zarr ERA5 năm {TARGET_YEAR} ---")

def save_zarr(ds, path, chunks=None):
    """Hàm tối ưu hóa xóa bỏ cấu trúc encoding cũ, chunking và ghi file Zarr"""
    for var in ds.data_vars:  ds[var].encoding = {}
    for coord in ds.coords:   ds[coord].encoding = {}
    if chunks:
        ds = ds.chunk(chunks)
    if os.path.exists(path):
        shutil.rmtree(path)
    ds.to_zarr(path, mode='w', consolidated=True)
    size = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(path) for f in fs) / 1e6
    print(f"  ✓ {os.path.basename(path)} -> Quy mô dung lượng: {size:.0f} MB")

# Lưu trữ trực tiếp tệp nén sử dụng khối SAVE_CHUNKS tối ưu
save_zarr(ds_x,     f'{OUT}/brain_{TARGET_YEAR}_x.zarr',     SAVE_CHUNKS)
save_zarr(ds_label, f'{OUT}/brain_{TARGET_YEAR}_label.zarr', SAVE_CHUNKS)
print(f"✓ Đã đồng bộ nhánh dữ liệu ERA5 năm {TARGET_YEAR} vào phân vùng lưu trữ.")

--- Tiến hành kết xuất luồng dữ liệu Zarr ERA5 năm 2025 ---


/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


  ✓ brain_2025_x.zarr -> Quy mô dung lượng: 3385 MB


/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


  ✓ brain_2025_label.zarr -> Quy mô dung lượng: 3385 MB
✓ Đã đồng bộ nhánh dữ liệu ERA5 năm 2025 vào phân vùng lưu trữ.


## Section 2: NASA — Align, Normalize, Save

In [8]:
print(f"--- Khởi tạo tiến trình tải dữ liệu NASA Raw .nc4 năm {TARGET_YEAR} ---")

ds_nasa = xr.open_mfdataset(
    f"{BASE_NASA}/nasa_ir_data/year_{TARGET_YEAR}/*.nc4",
    chunks=NASA_LOAD_CHUNKS,
    combine='by_coords',
    engine='h5netcdf'  # <--- BẠN THÊM DÒNG NÀY VÀO ĐÂY
)

# Đồng bộ hóa định danh biến nhiệt độ độ sáng Tb
if 'Tb' not in ds_nasa.data_vars and 'IR' in ds_nasa.data_vars:
     ds_nasa = ds_nasa.rename({'IR': 'Tb'})

print(f"  Cấu trúc các biến NASA : {list(ds_nasa.data_vars)}")
print(f"  Số Timesteps thu thập  : {len(ds_nasa.time)}")

# Thu hồi tài nguyên con trỏ ds_brain để giải phóng phân vùng RAM
del ds_brain
gc.collect()

--- Khởi tạo tiến trình tải dữ liệu NASA Raw .nc4 năm 2025 ---
  Cấu trúc các biến NASA : ['Tb']
  Số Timesteps thu thập  : 2920


0

In [9]:
print(f"--- Tiến hành căn chỉnh tọa độ NASA khớp với lưới của ERA5 năm {TARGET_YEAR} ---")

# 1. Căn chỉnh trục thời gian chính xác tương quan
try:
    da_nasa_aligned = ds_nasa['Tb'].sel(time=ds_x.time)
except KeyError:
    da_nasa_aligned = ds_nasa['Tb'].sel(time=ds_x.time, method='nearest')

print(f"  Số bước thời gian NASA Tb sau căn chỉnh: {len(da_nasa_aligned.time)}")

# 2. Căn chỉnh không gian địa lý (Nội suy tuyến tính độ phân giải cao 4km về lưới 0.25 độ)
print("  Đang thực thi thuật toán nội suy lưới không gian tuyến tính...")
eye_x = da_nasa_aligned.interp(lat=ds_x.lat, lon=ds_x.lon, method='linear')
print(f"  ✓ Kích thước ma trận chiều không gian sau nội suy: {dict(eye_x.sizes)}")

--- Tiến hành căn chỉnh tọa độ NASA khớp với lưới của ERA5 năm 2025 ---
  Số bước thời gian NASA Tb sau căn chỉnh: 1459
  Đang thực thi thuật toán nội suy lưới không gian tuyến tính...
  ✓ Kích thước ma trận chiều không gian sau nội suy: {'time': 1459, 'lat': 159, 'lon': 201}


In [10]:
print("--- Chuẩn hóa giá trị NASA Tb dựa trên bộ dữ liệu Stats toàn cục ---")

# Đọc cấu trúc NetCDF lưu trữ phân phối tĩnh của NASA
ds_nasa_stats = xr.open_dataset(f'{STATS_DIR}/nasa_normalization_stats.nc', engine='h5netcdf')
nasa_mean = float(ds_nasa_stats['mean'].values)
nasa_std  = float(ds_nasa_stats['std'].values)

print(f"  Tham số chuẩn hóa áp dụng: Mean={nasa_mean:.2f} K | Std={nasa_std:.2f} K")

# Chuẩn hóa Z-score ma trận dữ liệu vệ tinh
eye_x_norm = (eye_x - nasa_mean) / nasa_std

--- Chuẩn hóa giá trị NASA Tb dựa trên bộ dữ liệu Stats toàn cục ---
  Tham số chuẩn hóa áp dụng: Mean=275.57 K | Std=23.72 K


In [11]:
print(f"--- Tiến hành kết xuất dữ liệu cấu trúc vệ tinh NASA năm {TARGET_YEAR} ---")

save_zarr(eye_x_norm.to_dataset(name='Tb'), f'{OUT}/eye_{TARGET_YEAR}_norm.zarr', SAVE_CHUNKS)
print(f"✓ Nhánh dữ liệu ảnh mây vệ tinh NASA năm {TARGET_YEAR} đã được ghi thành công.")

--- Tiến hành kết xuất dữ liệu cấu trúc vệ tinh NASA năm 2025 ---


/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


  ✓ eye_2025_norm.zarr -> Quy mô dung lượng: 150 MB
✓ Nhánh dữ liệu ảnh mây vệ tinh NASA năm 2025 đã được ghi thành công.


## Section 3: Sanity Check (Kiểm định chất lượng dữ liệu đầu ra)

In [12]:
print("=" * 60)
print(f"TIẾN HÀNH KIỂM ĐỊNH TOÀN VẸN (SANITY CHECK) NĂM {TARGET_YEAR}")
print("=" * 60)

# Mở lại luồng dữ liệu Zarr vừa kết xuất để xác thực
_x = xr.open_zarr(f'{OUT}/brain_{TARGET_YEAR}_x.zarr',     consolidated=True)
_y = xr.open_zarr(f'{OUT}/brain_{TARGET_YEAR}_label.zarr', consolidated=True)
_e = xr.open_zarr(f'{OUT}/eye_{TARGET_YEAR}_norm.zarr',     consolidated=True)

# 1. Đánh giá tính liên kết trục thời gian X và Y
ok1 = (_x.time.values == _y.time.values).all()
print(f"  X/Y đồng bộ trục thời gian         : {'✓ CHUẨN' if ok1 else '❌ THẤT BẠI'} ({len(_x.time)} mẫu)")

# 2. Đánh giá tính liên kết trục thời gian X và Tập dữ liệu Vệ tinh (NASA Eye)
ok2 = (_x.time.values == _e.time.values).all()
print(f"  X/Eye đồng bộ trục thời gian       : {'✓ CHUẨN' if ok2 else '❌ THẤT BẠI'}")

# 3. Kiểm tra tính tương thích lưới tọa độ không gian (Grid matching)
ok3 = (_x.lat.values == _e.lat.values).all() and (_x.lon.values == _e.lon.values).all()
print(f"  X/Eye khớp lưới tọa độ không gian  : {'✓ CHUẨN' if ok3 else '❌ THẤT BẠI'}")

# 4. Kiểm toán phân phối chuẩn hóa sơ bộ (Thực hiện trên lát cắt mẫu của biến đầu tiên)
sample_var = list(_x.data_vars)[0]
m = float(_x[sample_var].isel(time=slice(0,50)).mean().compute())
s = float(_x[sample_var].isel(time=slice(0,50)).std().compute())
norm_ok = abs(m) < 0.6 and 0.4 < s < 2.5
print(f"  Phân phối chuẩn [{sample_var:s}] Mean={m:.3f} Std={s:.3f} : {'✓ HỢP LỆ' if norm_ok else '⚠️ LỆCH PHÂN PHỐI (Bình thường đối với mẫu cắt ngắn)'}")

# 5. Kiểm tra rò rỉ hoặc khuyết thiếu dữ liệu (Kiểm tra giá trị NaN tại t=0)
nan_x = int(_x[sample_var].isel(time=0).isnull().sum().compute())
nan_y = int(_y[sample_var].isel(time=0).isnull().sum().compute())
nan_e = int(_e['Tb'].isel(time=0).isnull().sum().compute())
print(f"  Thống kê NaN [t=0]: Tập X={nan_x}, Tập Y={nan_y}, Tập Vệ tinh Eye={nan_e}")

print("\n" + "=" * 60)
print("DANH SÁCH FILE THỰC TẾ TRÊN ĐĨA")
print("=" * 60)
all_outputs = [
    f'brain_{TARGET_YEAR}_x.zarr',
    f'brain_{TARGET_YEAR}_label.zarr',
    f'eye_{TARGET_YEAR}_norm.zarr'
]
for name in all_outputs:
    path = f'{OUT}/{name}'
    if os.path.exists(path):
        size = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(path) for f in fs) / 1e6
        print(f"  ✓ {name:<40s} Dung lượng: {size:6.0f} MB")
    else:
        print(f"  ❌ HỆ THỐNG THIẾU TẬP TIN: {name}")

print(f"\n--- Tiến trình tiền xử lý cho năm {TARGET_YEAR} hoàn tất kết quả an toàn. Sẵn sàng chuyển sang tệp tiếp theo ---")

TIẾN HÀNH KIỂM ĐỊNH TOÀN VẸN (SANITY CHECK) NĂM 2025
  X/Y đồng bộ trục thời gian         : ✓ CHUẨN (1459 mẫu)
  X/Eye đồng bộ trục thời gian       : ✓ CHUẨN
  X/Eye khớp lưới tọa độ không gian  : ✓ CHUẨN
  Phân phối chuẩn [d2m] Mean=-0.566 Std=1.566 : ✓ HỢP LỆ
  Thống kê NaN [t=0]: Tập X=0, Tập Y=0, Tập Vệ tinh Eye=285

DANH SÁCH FILE THỰC TẾ TRÊN ĐĨA
  ✓ brain_2025_x.zarr                        Dung lượng:   3385 MB
  ✓ brain_2025_label.zarr                    Dung lượng:   3385 MB
  ✓ eye_2025_norm.zarr                       Dung lượng:    150 MB

--- Tiến trình tiền xử lý cho năm 2025 hoàn tất kết quả an toàn. Sẵn sàng chuyển sang tệp tiếp theo ---
